# CS4168 Data Mining Project  
## Spotify Tracks Dataset Analysis

This notebook performs:
- Exploratory Data Analysis (EDA)
- Clustering (K-Means & DBSCAN)
- Classification (Predicting Popularity Category)
- Regression (Predicting Popularity Score)

To Activate .venv on Windows
-Set-ExecutionPolicy -Scope Process -ExecutionPolicy RemoteSigned   
-.venv\Scripts\Activate.ps1

In [2]:
# imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA

from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
# Load de Data

df = pd.read_csv("tracks2026.csv")
df.head()

# 1. Exploratory Data Analysis

In [ ]:
# Basic data inspect

df.info()
df.describe()
df.isnull().sum()

In [ ]:
# Popularity distribution

plt.figure(figsize=(6,4))
sns.histplot(df["popularity"], bins=30)
plt.title("Popularity Distribution")
plt.show()

In [ ]:
# Correlation Heatmap

plt.figure(figsize=(12,8))
sns.heatmap(df.corr(numeric_only=True), cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

# 2. Clustering Analysis

In [ ]:
# Data prep for Clustering

X_cluster = df.drop(columns=["track_genre"])
X_cluster = X_cluster.select_dtypes(include=np.number)1 

In [ ]:
# Elbow Method for K-Means

inertia = []

for k in range(2, 10):
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("kmeans", KMeans(n_clusters=k, random_state=42))
    ])
    pipe.fit(X_cluster)
    inertia.append(pipe.named_steps["kmeans"].inertia_)

plt.plot(range(2,10), inertia)
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.show()

In [ ]:
# Fit Final K-Means (Eg - k=3)

kmeans_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("kmeans", KMeans(n_clusters=3, random_state=42))
])

kmeans_pipe.fit(X_cluster)

labels = kmeans_pipe.named_steps["kmeans"].labels_

print("Silhouette Score:",
      silhouette_score(StandardScaler().fit_transform(X_cluster), labels))

In [ ]:
# Visualise Clusters (w/ PCS)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.scatter(X_pca[:,0], X_pca[:,1], c=labels)
plt.title("K-Means Clusters (PCA)")
plt.show()